# Reproduce a GSPC Living Board snapshot

Read the published rows and report coverage without mixing distinct measurement kinds. This notebook performs descriptive analysis of a snapshot. It does not rerun the underlying instruments or verify signatures.

Attach [the Kaggle dataset](https://www.kaggle.com/datasets/nicktempleman/csoai-gspc-living-board) as an input. To reproduce the same result, retain its version, archive the exact input bytes and compare the SHA-256 printed below. For local use, set `GSPC_SNAPSHOT` to an exact CSV or board API JSON file. No network or GPU is needed. See the [live board](https://councilof.ai/api/gspc) for the current inventory.

In [ ]:
import csv
import hashlib
import io
import json
import os
from pathlib import Path

explicit = os.environ.get("GSPC_SNAPSHOT")
if explicit:
    source = Path(explicit)
else:
    candidates = sorted(Path("/kaggle/input").rglob("gspc-axes.csv"))
    if len(candidates) != 1:
        raise ValueError("Attach exactly one GSPC CSV or set GSPC_SNAPSHOT to the file to inspect.")
    source = candidates[0]

raw = source.read_bytes()
digest = hashlib.sha256(raw).hexdigest()
expected = os.environ.get("GSPC_EXPECTED_SHA256")
if expected and digest != expected.lower():
    raise ValueError("Snapshot SHA-256 does not match the expected digest.")
if source.suffix.lower() == ".json":
    rows = json.loads(raw)["axes"]
elif source.suffix.lower() == ".csv":
    rows = list(csv.DictReader(io.StringIO(raw.decode("utf-8-sig"))))
else:
    raise ValueError("Use the board JSON or gspc-axes.csv.")

required = {"axis", "kind", "status"}
if not isinstance(rows, list) or not rows or any(
    not isinstance(row, dict) or not required <= row.keys()
    or not all(isinstance(row[k], str) and row[k].strip() for k in required)
    for row in rows
):
    raise ValueError("Missing or invalid axis, kind or status.")
if len({row["axis"] for row in rows}) != len(rows):
    raise ValueError("Axis identifiers must be unique.")
print(f"{source.name} | SHA-256 {digest} | {len(rows)} distinct axes")

## Compare like with like
Report model separation only within model comparisons. An empty numeric field stays unreported. A row with reported data describes its own instrument and observation.

In [ ]:
from collections import Counter

def present(value):
    return value is not None and value != ""

comparisons = [row for row in rows if row["kind"] == "model-comparison"]
print("Kind and reported state:", dict(sorted(Counter((r["kind"], r["status"]) for r in rows).items())))
print("Separation (model-comparison rows only):",
      dict(sorted(Counter(r.get("separation") or "UNREPORTED" for r in comparisons).items())))
print("Missing accuracy, by kind:",
      dict(sorted(Counter(r["kind"] for r in rows if not present(r.get("accuracy"))).items())))
print("axis | kind | status | sample size | accuracy | separation")
for row in sorted(rows, key=lambda r: (r["kind"], r["axis"])):
    fields = [row.get(k) for k in ("axis", "kind", "status", "n", "accuracy", "separation")]
    print(" | ".join(str(value) if present(value) else "unreported" for value in fields))